In [1]:
from moabb.paradigms import P300
from moabb.datasets import *

paradigm = P300(resample=48)
datasets = [BNCI2014_008(), BNCI2015_003()]


In [2]:
from sklearn.model_selection import StratifiedKFold
from hoda.hoda import BTTDA, GreedyBTTDA, HODA, trunc_eigh
from hoda.cov import mode_scatter, ledoit_wolf_shrinkage
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from mne.decoding import Scaler
import  warnings
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectFwe
from sklearn.preprocessing import StandardScaler, FunctionTransformer
import warnings
from joblib import parallel_backend
from joblib import Parallel
from hoda.classification import SelectF
from sklearn.pipeline import make_pipeline
import tensorly as tl

clf = make_pipeline(
    SelectF(alpha=.05),
    FunctionTransformer(tl.to_numpy),
    StandardScaler(),
    LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
)

bttda = GreedyBTTDA(
    max_blocks=2,
    truncate=False,
    hoda_params=dict(
        rank=None,
        max_iter=128,
        tol=1e-8,
        init ='random',
        shrinkage='lw',
        toeplitz=None,
        obj='tr',
        solver='lanczos',
        taper=False,
        extra_train_info=False,
        verbose=False,
        random_state=42,
        delta=None,
       
    ),
    verbose=True,
    extra_train_info=True,
    cv=StratifiedKFold(random_state=42, shuffle=True),
    clf=clf,
    scoring='roc_auc',
)

In [ ]:
import tensorly as tl
import pandas as pd

select_info = []
train_info = []
for dataset in datasets:
    for subject in dataset.subject_list[:1]:
        epochs, labels, meta = paradigm.get_data(
            dataset=dataset, 
             subjects=[subject],
             return_epochs=True
        )
        for session in meta['session'].unique()[:1]:
            idc = meta['session'] == session
            epochs_ses = epochs[idc]
            labels_ses = labels[idc]
            meta_ses = meta[idc]

            X = epochs_ses.get_data()
            X = tl.tensor(X)
            y = labels_ses

            print(f'dataset={dataset} subject={subject} session={session}')
            bttda.fit(X,y, test=False)
            subj_select_info = pd.DataFrame(bttda.model_select_info_) 
            subj_select_info['dataset'] = dataset.code
            subj_select_info['subject'] = subject
            subj_select_info['session'] = session
            subj_train_info = pd.DataFrame(bttda.train_info_)
            subj_train_info['dataset'] = dataset.code
            subj_train_info['subject'] = subject
            subj_train_info['session'] = session
            select_info.append(subj_select_info.reset_index())
            train_info.append(subj_train_info.reset_index())
select_info = pd.concat(select_info, ignore_index=True)
train_info = pd.concat(train_info, ignore_index=True)

/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 4200 events (all good), 0 – 1 s (baseline off), ~65.9 MB, data loaded,
 'Target': 700
 'NonTarget': 3500>
  warn(f"warnEpochs {epochs}")


Adding metadata with 3 columns
Adding metadata with 3 columns
4200 matching events found
No baseline correction applied
dataset=<moabb.datasets.bnci.BNCI2014_008 object at 0x14bc251f1010> subject=1 session=0
Model selection block 1/2...

Trying rank (1, 1)	

/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/paradigms/base.py:350: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  X = mne.concatenate_epochs(X)


score: 0.8071
Trying rank (2, 2)	score: 0.7937
Trying rank (4, 4)	score: 0.7670
Trying rank (8, 8)	score: 0.8022
Trying rank (8, 16)	score: 0.8287
Trying rank (8, 32)	score: 0.8282
Trying rank (8, 48)	score: 0.8300

Selected rank (8, 48) with score 0.8300
New ranks: [(8, 48)]




In [ ]:
select_info.to_csv('block_erp_select.csv')
select_info

In [ ]:
import seaborn as sns

idx = ['dataset', 'subject', 'session', 'block']
df = select_info.groupby(idx)[['train_score', 'val_score']].aggregate('mean')
df = df.melt(var_name='split', ignore_index=False)
df = df.reset_index()
sns.lineplot(data=df, x='block', y='value', style='split', hue='dataset')

In [ ]:
train_info.to_csv('block_erp_train.csv')
train_info

In [ ]:
df = train_info.groupby(idx)
df = df['nmse'].aggregate('mean')
df = df.reset_index()
sns.lineplot(data=df, x='block',y='nmse', hue='dataset')